<a href="https://colab.research.google.com/github/srepho/Allyanonimiser/blob/main/dspy_tutorials_expanded.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# DSPy Tutorial (Hands-on, provider-agnostic, no manual prompts)

**Goals for this session**  
- Program LLMs *without hand-written prompts* using **Signatures** and **Modules**.  
- **Switch providers** (OpenAI, Azure OpenAI, Anthropic, Gemini, local) with a single line change.  
- **Optimize** programs automatically with **GEPA** (reflective prompt evolution) and **BootstrapFewShot**.  
- Evaluate with simple, reproducible **metrics** and small **dev/train** sets like normal ML.

> This notebook is Colab-friendly. Bring at least one provider key. Azure users: set `AZURE_API_KEY`, `AZURE_API_BASE`, `AZURE_API_VERSION`, and `AZURE_DEPLOYMENT`.


In [ ]:

# ⬇️ One-time installs per runtime
!pip -q install -U dspy-ai gepa pandas datasets


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 121.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.7/261.7 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.4/278.4 kB 15.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.2 which is incompatible.
dask-cudf-cu12 25

In [ ]:
# 🔐 Load provider secrets from Colab userdata (no printing)
import os
try:
    from google.colab import userdata
except Exception:
    class _UD:
        def get(self, k):
            return os.getenv(k)
    userdata = _UD()

def _env_from_secret(name, envname=None):
    v = userdata.get(name)
    if v:
        os.environ[envname or name] = v

for name in [
    "OPEN_AI",
    "CLAUDE",
    "GEMINI",

]:
    _env_from_secret(name)

#OPENAI_MODEL_OVERRIDE    = userdata.get("OPENAI_MODEL")
#ANTHROPIC_MODEL_OVERRIDE = userdata.get("ANTHROPIC_MODEL")
#GEMINI_MODEL_OVERRIDE    = userdata.get("GEMINI_MODEL")

DEFAULT_MODELS = {
    "openai":    "openai/gpt-4o-mini",
    "anthropic": "anthropic/claude-3-5-sonnet-20241022",
    "gemini":    "gemini/gemini-2.0-pro-exp-02-05",
}
VENDOR_PREFIXES = {"openai": "openai/", "anthropic": "anthropic/", "gemini": "gemini/"}

def canonical_model(provider: str, maybe_model: str | None) -> str:
    if maybe_model and maybe_model.startswith(("sk-", "gsk_", "AIza")):
        maybe_model = None
    if not maybe_model:
        return DEFAULT_MODELS[provider]
    if not any(maybe_model.startswith(p) for p in VENDOR_PREFIXES.values()):
        maybe_model = VENDOR_PREFIXES[provider] + maybe_model
    return maybe_model


In [ ]:
import dspy, os

lm = dspy.LM(
    "openai/gpt-4.1",           # vendor-prefixed model id
    api_key=os.getenv("OPEN_AI")
)
dspy.configure(lm=lm)

from dspy import Predict

destination_summary = dspy.Predict("destination -> summary")
destination_summary(destination="Sydney")



Prediction(
    summary='The largest city in Australia is Sydney. Located on the east coast in the state of New South Wales, Sydney is renowned for its iconic landmarks such as the Sydney Opera House and Sydney Harbour Bridge. It is the most populous city in Australia, serving as a major financial, cultural, and economic hub for the country.'
)

In [ ]:
destination_summary = dspy.ChainOfThought("destination -> summary")
destination_summary(destination="Sydney")

Prediction(
    reasoning='Sydney is a vibrant, cosmopolitan city known for its iconic landmarks such as the Sydney Opera House and Harbour Bridge. It offers a mix of beautiful beaches, diverse dining options, and a lively arts and culture scene. Sydney is also a gateway to natural attractions like the Blue Mountains and coastal walks, making it an appealing destination for both urban exploration and outdoor activities.',
    summary='Sydney is a dynamic city famous for its landmarks, beaches, cultural attractions, and proximity to natural wonders, making it a top travel destination.'
)

In [ ]:
signature = 'sentence -> sentiment'
classify = dspy.Predict(signature)
sentence = "it's a charming and often affecting journey"
classify(sentence=sentence)

Prediction(
    sentiment='positive'
)

In [ ]:
lm.inspect_history(n=1)





[2025-09-11T05:29:44.834472]

System message:

Your input fields are:
1. `sentence` (str):
Your output fields are:
1. `sentiment` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## sentence ## ]]
{sentence}

[[ ## sentiment ## ]]
{sentiment}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `sentence`, produce the fields `sentiment`.


User message:

[[ ## sentence ## ]]
it's a charming and often affecting journey

Respond with the corresponding output fields, starting with the field `[[ ## sentiment ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## sentiment ## ]]
positive

[[ ## completed ## ]]







In [ ]:
class DestinationSummary(dspy.Signature):
    """Write a crisp, factual 2–3 sentence overview."""
    destination: str = dspy.InputField()
    summary: str = dspy.OutputField()

summarizer = dspy.Predict(DestinationSummary)
print(summarizer(destination="Sydney").summary)

Sydney, Australia’s largest city, is renowned for its iconic landmarks such as the Sydney Opera House and Harbour Bridge. The city offers a vibrant cultural scene, beautiful beaches like Bondi, and a diverse culinary landscape.


In [ ]:
from typing import Literal

class Triage(dspy.Signature):
    """Route support tickets. Labels ∈ {billing, technical, account}. Keep it short."""
    text: str = dspy.InputField()
    label: Literal['billing','technical','account'] = dspy.OutputField()
    rationale: str = dspy.OutputField(desc="one sentence why")

class TriageModule(dspy.Module):
    def __init__(self): self.classify = dspy.ChainOfThought(Triage)
    def forward(self, text: str):
        p = self.classify(text=text)
        return dspy.Prediction(label=p.label, rationale=p.rationale)

triage_program = TriageModule()

DATA = [
    ("Refund not processed, charged twice", "billing"),
    ("Can't reset password, link expired", "account"),
    ("App crashes when uploading PDF", "technical"),
    ("Wrong tax amount on invoice", "billing"),
    ("Two-factor code never arrives", "account"),
    ("API returns 500 for /v2/claims", "technical"),
]
train = [dspy.Example(text=t, label=y).with_inputs("text") for t,y in DATA[:4]]
dev   = [dspy.Example(text=t, label=y).with_inputs("text") for t,y in DATA[4:]]

def triage_metric(ex, pred, *_, **__):
    return float((pred.label or "").strip().lower() == ex.label)

from dspy.evaluate import Evaluate
baseline = Evaluate(devset=dev, metric=triage_metric, display_progress=False)(triage_program)
print(f"Baseline accuracy: {baseline.score:.1f}%")  # score is already a percentage

2025/09/11 05:11:40 INFO dspy.evaluate.evaluate: Average Metric: 1.0 / 2 (50.0%)


Baseline accuracy: 50.0%


In [ ]:
bf = dspy.BootstrapFewShot(
    metric=triage_metric,
    max_bootstrapped_demos=4,
    max_labeled_demos=8,
    max_rounds=1
)
triage_bf = bf.compile(student=triage_program, trainset=train)

post_bf = Evaluate(devset=dev, metric=triage_metric, display_progress=False)(triage_bf)
print(f"BootstrapFewShot accuracy: {post_bf.score:.1f}%")

100%|██████████| 4/4 [00:04<00:00,  1.08s/it]


Bootstrapped 4 full traces after 3 examples for up to 1 rounds, amounting to 4 attempts.


2025/09/11 05:12:06 INFO dspy.evaluate.evaluate: Average Metric: 2.0 / 2 (100.0%)


BootstrapFewShot accuracy: 100.0%


In [ ]:
class Feedback(dict):
    """Dict that also supports attribute access: fb.score, fb.feedback"""
    __getattr__ = dict.get
    def __setattr__(self, k, v): self[k] = v

def triage_metric_gepa(ex, pred, trace=None, pred_name=None, pred_trace=None):
    gold = ex.label
    got  = (pred.label or "").strip().lower()
    score = float(got == gold)  # 0.0 or 1.0 (keep identical to module-level score)

    # Program-level eval → return a scalar float
    if pred_name is None:
        return score

    # Predictor-level feedback → return score + text (as a dot-dict)
    if score == 1.0:
        fb_text = "Correct. Keep rules concise."
    else:
        fb_text = ("Misclassified. billing⇢ invoice/refund/charge/card; "
                   "account⇢ password/2FA/email change; "
                   "technical⇢ crash/500/upload failure.")
    return Feedback(score=score, feedback=fb_text)



gepa = dspy.GEPA(metric=triage_metric_gepa, auto="light", reflection_lm=dspy.settings.lm)
triage_gepa = gepa.compile(student=triage_program, trainset=train)

post_gepa = Evaluate(devset=dev, metric=triage_metric, display_progress=False)(triage_gepa)
print(f"GEPA accuracy: {post_gepa.score:.1f}%")

2025/09/11 05:14:15 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 396 metric calls of the program. This amounts to 99.00 full evals on the train set.
2025/09/11 05:14:15 INFO dspy.teleprompt.gepa.gepa: Using 4 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget.
GEPA Optimization:   0%|          | 0/396 [00:00<?, ?rollouts/s]2025/09/11 05:14:15 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 4 (75.0%)
2025/09/11 05:14:15 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.75
2025/09/11 05:14:15 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 0.75


Average Metric: 2.00 / 3 (66.7%): 100%|██████████| 3/3 [00:00<00:00, 247.60it/s]

2025/09/11 05:14:16 INFO dspy.evaluate.evaluate: Average Metric: 2.0 / 3 (66.7%)


2025/09/11 05:14:18 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Proposed new text for classify.predict: You are given support ticket texts. Your task is to assign a single label to each ticket, choosing from the set: {billing, technical, account}. Keep your response short: output only the label, no explanations or reasoning.

Use the following rules to determine the correct label:
- billing: Issues related to invoices, refunds, charges, payments, or credit/debit cards.
- account: Issues related to passwords, two-factor authentication (2FA), email changes, or other account access/management problems.
- technical: Issues related to app crashes, error codes (e.g., 500 errors), upload failures, or other software malfunctions.

Do not provide any additional text or justification—just the label.
2025/09/11 05:14:19 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2025/09/11 05:14:20 INFO dspy.evaluate.evaluate: Average Metric: 4.0 / 4 (100.0%)
2025/09/11 05:14:20 INFO dspy.tele

Average Metric: 3.00 / 3 (100.0%): 100%|██████████| 3/3 [00:00<00:00, 294.56it/s]

...

2025/09/11 05:14:26 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2025/09/11 05:14:26 INFO dspy.teleprompt.gepa.gepa: Iteration 128: All subsample scores perfect. Skipping.
2025/09/11 05:14:26 INFO dspy.teleprompt.gepa.gepa: Iteration 128: Reflective mutation did not propose a new candidate
2025/09/11 05:14:26 INFO dspy.teleprompt.gepa.gepa: Iteration 129: Selected program 1 score: 1.0



Average Metric: 3.00 / 3 (100.0%): 100%|██████████| 3/3 [00:00<00:00, 677.37it/s]

2025/09/11 05:14:26 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)


2025/09/11 05:14:26 INFO dspy.teleprompt.gepa.gepa: Iteration 129: All subsample scores perfect. Skipping.
2025/09/11 05:14:26 INFO dspy.teleprompt.gepa.gepa: Iteration 129: Reflective mutation did not propose a new candidate
GEPA Optimization: 100%|█████████▉| 395/396 [00:10<00:00, 36.79rollouts/s]
2025/09/11 05:14:27 INFO dspy.evaluate.evaluate: Average Metric: 2.0 / 2 (100.0%)


GEPA accuracy: 100.0%


In [ ]:
def extract_prompt(entry: dict) -> str:
    p = entry.get("prompt")
    if isinstance(p, str) and p.strip(): return p
    msgs = entry.get("messages")
    if isinstance(msgs, list) and msgs:
        lines = []
        for m in msgs:
            role = (m.get("role") or "system").upper()
            content = m.get("content", "")
            if isinstance(content, list):
                content = "".join([part.get("text","") if isinstance(part, dict) else str(part) for part in content])
            lines.append(f"{role}: {content}")
        return "\n".join(lines)
    kw = entry.get("kwargs", {})
    for k in ("input","inputs","prompt"):
        if isinstance(kw.get(k), str) and kw[k].strip(): return kw[k]
    return "<no prompt found>"

_ = destination_summary(destination="Melbourne")
print(extract_prompt(dspy.settings.lm.history[-1]))

SYSTEM: Your input fields are:
1. `destination` (str):
Your output fields are:
1. `summary` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## destination ## ]]
{destination}

[[ ## summary ## ]]
{summary}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `destination`, produce the fields `summary`.
USER: [[ ## destination ## ]]
Melbourne

Respond with the corresponding output fields, starting with the field `[[ ## summary ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


In [ ]:
def format_usd(x):
    if x is None: return "n/a"
    if x >= 1:    return f"${x:,.2f}"
    if x >= 0.01: return f"${x:,.4f} ({x*100:.2f}¢)"
    micro = x * 1_000_000; cents = x * 100
    return f"${x:.6f} ({cents:.4f}¢; {micro:.0f} μ$)"

def pretty_last_call(lm):
    h = lm.history[-1]; u = h.get("usage", {}) or {}
    pt, ct = u.get("prompt_tokens",0) or 0, u.get("completion_tokens",0) or 0
    tt = u.get("total_tokens", pt+ct)
    cost = h.get("cost")
    print(f"Model: {h.get('model','?')}")
    print(f"Tokens — prompt: {pt:,} | completion: {ct:,} | total: {tt:,}")
    print(f"Cost: {format_usd(cost)}")
    if cost and tt:
        print(f"Effective rate: {format_usd(cost/tt*1000)} per 1K tok (approx)")

_ = dspy.settings.lm("cache me!", temperature=0.0)
_ = dspy.settings.lm("cache me!", temperature=0.0)  # likely cache hit
pretty_last_call(dspy.settings.lm)

Model: openai/gpt-4.1
Tokens — prompt: 0 | completion: 0 | total: 0
Cost: $0.001404 (0.1404¢; 1404 μ$)


In [ ]:
# Save state-only JSON (nice for git): instructions + demos
triage_gepa.save("/content/triage_gepa.json", save_program=False)

# Fresh runtime / later:
loaded = TriageModule()
loaded.load("/content/triage_gepa.json")
res = Evaluate(devset=dev, metric=triage_metric, display_progress=False)(loaded)
print(f"Reloaded program accuracy: {res.score:.1f}%")

2025/09/11 05:17:53 INFO dspy.evaluate.evaluate: Average Metric: 2.0 / 2 (100.0%)


Reloaded program accuracy: 100.0%



## Part A — *Program, don't prompt:* a tiny triage classifier

We define a **Signature** (schema) and a **Module**. No prompt strings.  
DSPy uses the signature to build the right prompt behind the scenes.


In [ ]:

# 1) Define the task as a Signature (inputs/outputs) — no prompt text.
class Triage(dspy.Signature):
    """Route support tickets. Labels ∈ {billing, technical, account}. Keep it short."""
    text: str = dspy.InputField()
    label: Literal['billing','technical','account'] = dspy.OutputField()
    rationale: str = dspy.OutputField(desc="one sentence justification")

# 2) Wrap it in a Module — we'll use ChainOfThought to encourage reasoning internally.
class TriageModule(dspy.Module):
    def __init__(self):
        self.classify = dspy.ChainOfThought(Triage)
    def forward(self, text: str):
        pred = self.classify(text=text)
        return dspy.Prediction(label=pred.label, rationale=pred.rationale)

triage_program = TriageModule()


In [ ]:
def triage_metric_gepa(ex, pred, trace=None, pred_name=None, pred_trace=None):
    gold = ex.label
    got  = (pred.label or "").strip().lower()
    score = float(got == gold)
    if pred_name is None:
        return score
    if score == 1.0:
        fb = "Correct. Label matches. Keep instructions concise."
    else:
        fb = (
            f"Misclassified. Expected '{gold}', got '{got}'. "
            "Guidance: billing ⇢ invoice, refund, charge, credit card; "
            "account ⇢ password, 2FA, email change; "
            "technical ⇢ crash, 500 errors, upload failures."
        )
    return {"score": score, "feedback": fb}

# (Replaced old triage_metric_with_feedback with triage_metric_gepa)


In [ ]:

# Baseline (no optimization)
from dspy.evaluate import Evaluate
evaluator = Evaluate(devset=devset, metric=triage_metric, display_progress=False)
baseline = evaluator(triage_program)
print(f"Baseline accuracy: {baseline.score:.2f}%")


2025/09/11 04:53:21 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 5 (60.0%)


Baseline accuracy: 60.00%


In [ ]:
# ⛏️ Optimizer 1: BootstrapFewShot — automatically builds few-shot demos.
from dspy.evaluate import Evaluate

bf = dspy.BootstrapFewShot(
    metric=triage_metric,
    max_bootstrapped_demos=4,
    max_labeled_demos=8,
    max_rounds=1
)

triage_bf = bf.compile(student=triage_program, trainset=trainset)
post_bf = Evaluate(devset=devset, metric=triage_metric, display_progress=False)(triage_bf)
print(f"BootstrapFewShot accuracy: {post_bf.score:.1f}%")



 57%|█████▋    | 4/7 [00:00<00:00, 11.37it/s]
2025/09/11 04:53:39 INFO dspy.evaluate.evaluate: Average Metric: 5.0 / 5 (100.0%)


Bootstrapped 4 full traces after 4 examples for up to 1 rounds, amounting to 4 attempts.
BootstrapFewShot accuracy: 100.0%


In [ ]:
triage_bf.history[-1]

{'prompt': None,
 'messages': [{'role': 'system',
   'content': "Your input fields are:\n1. `text` (str):\nYour output fields are:\n1. `reasoning` (str): \n2. `label` (Literal['billing', 'technical', 'account']): \n3. `rationale` (str): one sentence justification\nAll interactions will be structured in the following way, with the appropriate values filled in.\n\n[[ ## text ## ]]\n{text}\n\n[[ ## reasoning ## ]]\n{reasoning}\n\n[[ ## label ## ]]\n{label}        # note: the value you produce must exactly match (no extra characters) one of: billing; technical; account\n\n[[ ## rationale ## ]]\n{rationale}\n\n[[ ## completed ## ]]\nIn adhering to this structure, your objective is: \n        Route support tickets. Labels ∈ {billing, technical, account}. Keep it short."},
  {'role': 'user',
   'content': 'This is an example of the task, though some input or output fields are not supplied.\n\n[[ ## text ## ]]\nFile parsing throws stack trace'},
  {'role': 'assistant',
   'content': '[[ ## rea

In [ ]:
# ⛏️ Optimizer 2: GEPA — reflective prompt evolution with textual feedback.
# Use a metric that returns a float at program-level and {score, feedback} at predictor-level.
gepa = dspy.GEPA(metric=triage_metric_gepa, auto="light", reflection_lm=dspy.settings.lm)
triage_gepa = gepa.compile(student=triage_program, trainset=trainset)
post_gepa = Evaluate(devset=devset, metric=triage_metric, display_progress=False)(triage_gepa)
print(f"GEPA accuracy: {post_gepa.score:.1f}%")


2025/09/11 03:44:23 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 408 metric calls of the program. This amounts to 58.29 full evals on the train set.
2025/09/11 03:44:23 INFO dspy.teleprompt.gepa.gepa: Using 7 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget.
GEPA Optimization:   0%|          | 0/408 [00:00<?, ?rollouts/s]2025/09/11 03:44:23 INFO dspy.evaluate.evaluate: Average Metric: 7.0 / 7 (100.0%)
2025/09/11 03:44:23 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 1.0
2025/09/11 03:44:23 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 1.0


Average Metric: 3.00 / 3 (100.0%): 100%|██████████| 3/3 [00:00<00:00, 171.61it/s]

2025/09/11 03:44:23 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2025/09/11 03:44:23 INFO dspy.teleprompt.gepa.gepa: Iteration 1: All subsample scores perfect. Skipping.
2025/09/11 03:44:23 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Reflective mutation did not propose a new candidate
GEPA Optimization:   2%|▏         | 10/408 [00:00<00:05, 71.75rollouts/s]2025/09/11 03:44:23 INFO dspy.teleprompt.gepa.gepa: Iteration 2: Selected program 0 score: 1.0



Average Metric: 3.00 / 3 (100.0%): 100%|██████████| 3/3 [00:00<00:00, 120.96it/s]

2025/09/11 03:44:23 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2025/09/11 03:44:23 INFO dspy.teleprompt.gepa.gepa: Iteration 2: All subsample scores perfect. Skipping.
2025/09/11 03:44:23 INFO dspy.teleprompt.gepa.gepa: Iteration 2: Reflective mutation did not propose a new candidate
2025/09/11 03:44:23 INFO dspy.teleprompt.gepa.gepa: Iteration 3: Selected program 0 score: 1.0


...

2025/09/11 03:44:32 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2025/09/11 03:44:32 INFO dspy.teleprompt.gepa.gepa: Iteration 133: All subsample scores perfect. Skipping.
2025/09/11 03:44:32 INFO dspy.teleprompt.gepa.gepa: Iteration 133: Reflective mutation did not propose a new candidate
2025/09/11 03:44:32 INFO dspy.teleprompt.gepa.gepa: Iteration 134: Selected program 0 score: 1.0



Average Metric: 3.00 / 3 (100.0%): 100%|██████████| 3/3 [00:00<00:00, 121.35it/s]

2025/09/11 03:44:32 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2025/09/11 03:44:32 INFO dspy.teleprompt.gepa.gepa: Iteration 134: All subsample scores perfect. Skipping.
2025/09/11 03:44:32 INFO dspy.teleprompt.gepa.gepa: Iteration 134: Reflective mutation did not propose a new candidate
GEPA Optimization: 100%|█████████▉| 406/408 [00:09<00:00, 42.34rollouts/s]


2025/09/11 03:44:32 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 5 (60.0%)


GEPA accuracy: 60.0%



## Part B — Structured extraction (no regex rules, no prompt prose)

Define fields → get structured outputs. DSPy handles the prompting under the hood.


In [ ]:

class InvoiceFields(dspy.Signature):
    """Extract vendor, amount, and ISO date from a 1-line invoice blurb."""
    text: str = dspy.InputField()
    vendor: str = dspy.OutputField()
    amount: float = dspy.OutputField()
    date: str = dspy.OutputField(desc="YYYY-MM-DD if present, else 'unknown'")

extractor = dspy.Predict(InvoiceFields)

rows = [
    "Acme Corp invoice 1048 for $1299.50 dated 2025-07-15",
    "Charge of $89 from QuickHost, no date provided",
    "Contoso billed 2450 USD on 2025/08/01 for Q3 services",
]
for r in rows:
    pred = extractor(text=r)
    print(pred)


Prediction(
    vendor='Acme Corp',
    amount=1299.5,
    date='2025-07-15'
)
Prediction(
    vendor='QuickHost',
    amount=89.0,
    date='unknown'
)
Prediction(
    vendor='Contoso',
    amount=2450.0,
    date='2025-08-01'
)



## Part C — Swap providers in one line

We keep the *same* program and evaluation set, then compare results while switching providers.
(You only need keys for the providers you want to test.)


## Part C² — Provider swap (Colab secrets–aware, safe models)

This cell compares baseline vs **BootstrapFewShot** after swapping providers, using model
overrides from Colab Secrets when provided (and falling back to sensible defaults).
It also **guards against accidentally passing API keys as model names** and prints
the actual routed `lm.model`.


In [ ]:
# 🧪 Provider matrix using Colab secrets + safe model canonicalization
from dspy.evaluate import Evaluate
import os
import pandas as pd

def provider_matrix():
    pt = [
        ("openai",    canonical_model("openai",    OPENAI_MODEL_OVERRIDE)),
        ("anthropic", canonical_model("anthropic", ANTHROPIC_MODEL_OVERRIDE)),
        ("gemini",    canonical_model("gemini",    GEMINI_MODEL_OVERRIDE)),
    ]
    # Include Azure when secrets are present (make_lm reads env for deployment)
    if os.getenv("AZURE_API_KEY") and os.getenv("AZURE_DEPLOYMENT"):
        pt.insert(1, ("azure", None))
    return pt

providers_to_try = provider_matrix()
results = []

for prov, model in providers_to_try:
    try:
        lm = make_lm(prov, model)  # uses env set from Colab secrets

        # Guard: ensure nobody accidentally passed an API key as a model id
        tail = lm.model.split("/", 1)[-1]
        if tail.startswith(("sk-", "gsk_", "AIza")):
            raise ValueError("Looks like an API key was passed as a model id. Check your secrets.")

        with dspy.context(lm=lm):
            prog = TriageModule()

            # Baseline
            base_eval = Evaluate(devset=devset, metric=triage_metric, display_progress=False)(prog)
            base = base_eval.score  # already a percent (0-100)

            # Optimized (no valset in compile)
            bfs_prog = dspy.BootstrapFewShot(
                metric=triage_metric,
                max_bootstrapped_demos=4,
                max_labeled_demos=8,
                max_rounds=1
            ).compile(student=prog, trainset=trainset)

            opt_eval = Evaluate(devset=devset, metric=triage_metric, display_progress=False)(bfs_prog)
            opt = opt_eval.score

            results.append({
                "provider": prov,
                "model": lm.model,
                "baseline_%": base,
                "after_BFS_%": opt,
            })
            print(f"{prov:9s}  model={lm.model:35s}  baseline={base:5.1f}%  after_BFS={opt:5.1f}%")

    except Exception as e:
        print(f"{prov:9s}  (skipped) -> {e.__class__.__name__}: {e}")

df_results = pd.DataFrame(results)
try:
    from caas_jupyter_tools import display_dataframe_to_user
    display_dataframe_to_user("Provider comparison", df_results)
except Exception:
    display(df_results)


In [ ]:

providers_to_try = [
    ("openai", userdata.get('OPEN_AI')),
    ("anthropic", userdata.get('CLAUDE')),
    ("gemini", userdata.get('GEMINI')),
]

from dspy.evaluate import Evaluate
results = []
for prov, model in providers_to_try:
    try:
        with dspy.context(lm=make_lm(prov, model)):
            prog = TriageModule()
            base = Evaluate(devset=devset, metric=triage_metric, display_progress=False)(prog).score
            opt  = Evaluate(devset=devset, metric=triage_metric, display_progress=False)(
                dspy.BootstrapFewShot(metric=triage_metric, max_bootstrapped_demos=4, max_labeled_demos=8, max_rounds=1
            ).compile(student=prog, trainset=trainset)).score
            results.append((prov, base, opt))
            print(f"{prov:9s}  baseline={base:5.1f}%  after_BFS={opt:5.1f}%")
    except Exception as e:
        print(f"{prov:9s}  (skipped) -> {e.__class__.__name__}: {e}")
results


2025/09/11 03:49:16 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 5 (60.0%)


openai     (skipped) -> TypeError: BootstrapFewShot.compile() got an unexpected keyword argument 'valset'


2025/09/11 03:49:22 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2025/09/11 03:49:22 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2025/09/11 03:49:22 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2025/09/11 03:49:22 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2025/09/11 03:49:22 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2025/09/11 03:49:25 ERROR dspy.utils.parallelizer: Error for Example({'text': 'My invoice shows wrong tax amount', 'label': 'billing'}) (input_keys={'text'}): litellm.APIError: AzureException APIError - argument of type 'NoneType' is not iterable. Set `provide_traceback=True` for traceback.
2025/09/11 03:49:25 ERROR dspy.utils.parallelizer: Error for Example({'text': 'Refund not pro

azure      (skipped) -> TypeError: BootstrapFewShot.compile() got an unexpected keyword argument 'valset'


2025/09/11 03:49:32 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2025/09/11 03:49:32 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2025/09/11 03:49:32 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2025/09/11 03:49:32 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2025/09/11 03:49:32 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2025/09/11 03:49:35 ERROR dspy.utils.parallelizer: Error for Example({'text': 'Refund not processed, double charged on my visa', 'label': 'billing'}) (input_keys={'text'}): litellm.AuthenticationError: Missing Anthropic API Key - A call is being made to anthropic but no key is set either in the environment variables or via params. Please set `ANTHROPIC_API_KEY` in your environment v

anthropic  (skipped) -> TypeError: BootstrapFewShot.compile() got an unexpected keyword argument 'valset'


2025/09/11 03:49:42 ERROR dspy.utils.parallelizer: Error for Example({'text': 'My invoice shows wrong tax amount', 'label': 'billing'}) (input_keys={'text'}): litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=AIzaSyAQW3ym
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers. Set `provide_traceback=True` for traceback.
2025/09/11 03:49:42 ERROR dspy.utils.parallelizer: Error for Example({'text': 'User role is incorrect for my login', 'label': 'account'}) (input_keys={'text'}): litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=AIzaSyAQW3ym
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers. Set `provide_traceback=True` for traceback.


gemini     (skipped) -> TypeError: BootstrapFewShot.compile() got an unexpected keyword argument 'valset'


[]

In [ ]:
import os

# however you store them:
os.environ["OPENAI_API_KEY"]    = userdata["OPENAI_API_KEY"]     # NOT userdata["OPEN_AI"]
os.environ["ANTHROPIC_API_KEY"] = userdata["ANTHROPIC_API_KEY"]  # NOT userdata["CLAUDE"]
os.environ["GEMINI_API_KEY"]    = userdata["GEMINI_API_KEY"]     # NOT userdata["GEMINI"]


from dspy.evaluate import Evaluate

results = []
for prov, model in providers_to_try:
    try:
        lm = make_lm(prov, model)     # your helper
        # sanity: catch “key passed as model” mistakes
        if lm.model.split("/", 1)[-1].startswith(("sk-", "gsk_", "AIza")):
            raise ValueError("Looks like you passed an API key as the model id.")

        with dspy.context(lm=lm):
            prog = TriageModule()

            base = Evaluate(devset=devset, metric=triage_metric, display_progress=False)(prog).score

            bfs_prog = dspy.BootstrapFewShot(
                metric=triage_metric,
                max_bootstrapped_demos=4,
                max_labeled_demos=8,
                max_rounds=1
            ).compile(student=prog, trainset=trainset)

            opt = Evaluate(devset=devset, metric=triage_metric, display_progress=False)(bfs_prog).score

            results.append((prov, lm.model, base, opt))
            print(f"{prov:9s}  model={lm.model:35s}  baseline={base:5.1f}%  after_BFS={opt:5.1f}%")

    except Exception as e:
        print(f"{prov:9s}  (skipped) -> {e.__class__.__name__}: {e}")

TypeError: 'module' object is not subscriptable


## Part D — Observability & caching

- Inspect token/cost via `lm.history`  
- Use `rollout_id` to force fresh calls while keeping caching on  


In [ ]:
def format_usd(x: float | None) -> str:
    """Human-readable USD with sensible units for tiny costs."""
    if x is None:
        return "n/a"
    if x >= 1:
        return f"${x:,.2f}"
    if x >= 0.01:
        return f"${x:,.4f} ({x*100:.2f}¢)"
    # Very small: show dollars + cents + microdollars
    micro = x * 1_000_000
    cents = x * 100
    return f"${x:.6f} ({cents:.4f}¢; {micro:.0f} μ$)"

def pretty_last_call(lm):
    """Print the last LM call with nice cost + token stats."""
    h = lm.history[-1]
    usage = h.get("usage", {}) or {}
    cost  = h.get("cost")
    model = h.get("model", "unknown")

    pt = usage.get("prompt_tokens", 0) or 0
    ct = usage.get("completion_tokens", 0) or 0
    tt = usage.get("total_tokens", pt + ct)

    lines = [
        f"Model: {model}",
        f"Tokens — prompt: {pt:,} | completion: {ct:,} | total: {tt:,}",
        f"Cost: {format_usd(cost)}"
    ]
    if cost is not None and tt:
        eff = cost / tt * 1000.0
        lines.append(f"Effective rate: {format_usd(eff)} per 1K tokens (approx)")

    print("\n".join(lines))

def pretty_totals(lm):
    """Aggregate cost/tokens across this session's history."""
    total_cost = sum((h.get("cost") or 0.0) for h in lm.history)
    total_tokens = sum((h.get("usage", {}).get("total_tokens") or 0) for h in lm.history)
    print(f"Session totals — tokens: {total_tokens:,}, cost: {format_usd(total_cost)}")
    if total_cost and total_tokens:
        eff = total_cost / total_tokens * 1000.0
        print(f"Session effective rate: {format_usd(eff)} per 1K tokens (approx)")


In [ ]:
def format_usd(x: float | None) -> str:
    if x is None:
        return "n/a"
    if x >= 1:
        return f"${x:,.2f}"
    if x >= 0.01:
        return f"${x:,.4f} ({x*100:.2f}¢)"
    micro = x * 1_000_000
    cents = x * 100
    return f"${x:.6f} ({cents:.4f}¢; {micro:.0f} μ$)"

def pretty_last_call(lm):
    h = lm.history[-1]
    usage = h.get("usage", {}) or {}
    cost  = h.get("cost")
    model = h.get("model", "unknown")
    pt = usage.get("prompt_tokens", 0) or 0
    ct = usage.get("completion_tokens", 0) or 0
    tt = usage.get("total_tokens", pt + ct)
    print(f"Model: {model}")
    print(f"Tokens — prompt: {pt:,} | completion: {ct:,} | total: {tt:,}")
    print(f"Cost: {format_usd(cost)}")
    if cost is not None and tt:
        eff = cost / tt * 1000.0
        print(f"Effective rate: {format_usd(eff)} per 1K tokens (approx)")

def pretty_totals(lm):
    total_cost = sum((h.get("cost") or 0.0) for h in lm.history)
    total_tokens = sum((h.get("usage", {}).get("total_tokens") or 0) for h in lm.history)
    print(f"Session totals — tokens: {total_tokens:,}, cost: {format_usd(total_cost)}")
    if total_cost and total_tokens:
        eff = total_cost / total_tokens * 1000.0
        print(f"Session effective rate: {format_usd(eff)} per 1K tokens (approx)")


In [ ]:

lm = dspy.settings.lm
_ = lm("Say this is cached!", temperature=0.0)
_ = lm("Say this is cached!", temperature=0.0)  # likely cache hit

print("Last call metadata keys:", list(lm.history[-1].keys()))
print("Approx cost (if supported):", lm.history[-1].get("cost"))
pretty_last_call(lm)
pretty_totals(lm)
# Forcing a fresh call while keeping cache on: change rollout_id + temperature > 0
predict = dspy.Predict("question -> answer")
a1 = predict(question="What is 1+1?", config={"rollout_id": 1, "temperature": 1.0})
a2 = predict(question="What is 1+1?", config={"rollout_id": 2, "temperature": 1.0})
print(a1.answer, "|", a2.answer)


Last call metadata keys: ['prompt', 'messages', 'kwargs', 'response', 'outputs', 'usage', 'cost', 'timestamp', 'uuid', 'model', 'response_model', 'model_type']
Approx cost (if supported): 5.6e-05
Model: openai/gpt-4.1
Tokens — prompt: 0 | completion: 0 | total: 0
Cost: $0.000056 (0.0056¢; 56 μ$)
Session totals — tokens: 12,199, cost: $0.8568 (85.68¢)
Session effective rate: $0.0702 (7.02¢) per 1K tokens (approx)
1+1 equals 2. | 1+1 equals 2.



## ✍️ Exercise (10–15 min)

1. Replace the toy `DATA` with a **slice of your real tickets**.  
2. Write a **stricter metric** (e.g., penalize wrong label *and* weak rationale).  
3. Swap the optimizer to **`dspy.MIPROv2`** or **`dspy.KNNFewShot`** and compare.  
4. Try Azure vs OpenAI vs Anthropic/Gemini at your org. Do results differ? Why?



---

### References & docs
- DSPy Language Models (provider switching): https://dspy.ai/learn/programming/language_models/  
- GEPA Optimizer: https://dspy.ai/api/optimizers/GEPA/  
- BootstrapFewShot: https://dspy.ai/api/optimizers/BootstrapFewShot/  
- Evaluation (metrics & Evaluate): https://dspy.ai/api/evaluation/Evaluate/  


## Save / Load the compiled program

In [ ]:
# 💾 Save/load the optimized program for next time
# After optimization (e.g., triage_gepa), save state-only JSON:
triage_gepa.save("/content/triage_gepa.json", save_program=False)

# Later / fresh runtime:
loaded = TriageModule()
loaded.load("/content/triage_gepa.json")
res = Evaluate(devset=devset, metric=triage_metric, display_progress=False)(loaded)
print(f"Reloaded program accuracy: {res.score:.1f}%")
